## Problem Statement

We all know that Health care is very important domain in the market. It is directly linked with the life 
of the individual; hence we have to be always be proactive in this particular domain. Money plays a 
major role in this domain, because sometime treatment becomes super costly and if any individual is 
not covered under the insurance then it will become a pretty tough financial situation for that 
individual. The companies in the medical insurance also want to reduce their risk by optimizing the 
insurance cost, because we all know a healthy body is in the hand of the individual only. If individual 
eat healthy and do proper exercise the chance of getting ill is drastically reduced. 


## Objective: 
The objective of this exercise is to build a model, using data that provide the 
optimum insurance cost for an individual. You have to use the health and habit related parameters for 
the estimated cost of insurance 

## Data Description

* **applicant_id:** Applicant unique ID
* **years_of_insurance_with_us:** Since how many years customer is taking policy from the same company only
* **regular_checkup_lasy_year:** Number of times customers has done the regular health check up in last one year
* **adventure_sports:** Customer is involved with adventure sports like climbing, diving etc.
* **Occupation:** Occupation of the customer
* **visited_doctor_last_1_year:** Number of times customer has visited doctor in last one year
* **Cholesterol_level:** Cholesterol level of the customers while applying for insurance
* **daily_avg_steps:** Average daily steps walked by customers
* **age:** Age of the customer
* **heart_decs_history:** Any past heart diseases
* **other_major_decs_history:** Any past major diseases apart from heart like any operation
* **Gender:** Gender of the customer
* **avg_glucose_level:** Average glucose level of the customer while applying the insurance
* **bmi:** BMI of the customer while applying the insurance
* **smoking_status:** Smoking status of the customer
* **Year_last_admitted:** When customer have been admitted in the hospital last time
* **Location:** Location of the hospital
* **weight:** Weight of the customer
* **covered_by_any_other_company:** Customer is covered from any other insurance company
* **Alcohol:** Alcohol consumption status of the customer
* **exercise:** Regular exercise status of the customer
* **weight_change_in_last_one_year:** How much variation has been seen in the weight of the customer in last year
* **fat_percentage:** Fat percentage of the customer while applying the insurance
* **insurance_cost:** Total Insurance cost

In [ ]:
## Importing libraries
import pandas as pd
import numpy as np
import datetime as dt

# for visualizing data
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# to scale the data using z-score
from sklearn.preprocessing import StandardScaler

# to compute distances
from scipy.spatial.distance import cdist, pdist

# to perform k-means clustering and compute silhouette scores
from sklearn.cluster import KMeans

from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor
from kmodes.kprototypes import KPrototypes



In [ ]:
## Reading Data
Data = pd.read_csv('Data.csv')

In [ ]:
# copying data to another variable to avoid any changes to original data
data = Data.copy()

## **Data Overview**

In [ ]:
data.head(5)

In [ ]:
data.tail(5)

In [ ]:
data.shape

 #### The dataset has 25000 rows and 24 columns.

### Checking the data types of the columns for the dataset.

In [ ]:
data.info()

### Checking for missing values

In [ ]:
# checking for null values
data.isnull().sum()

* There are missing values in **BMI** and **Year_last_admitted** attributes 

## Checking for Duplicates

In [ ]:
data.duplicated().sum()

* No duplicates in the dataset.

## Dropping **applicant_id** attribute as it doesnot contribute any value to in Data understanding stage

In [ ]:
data.drop(["applicant_id"], axis=1, inplace=True)

In [ ]:
data.head(5)

In [ ]:
for i in data.describe(include=["object"]).columns:
    print("Unique values in", i, "are :")
    print(data[i].value_counts())
    print("*" * 50)

In [ ]:
data.describe(include="all").T

### Observations:

**years_of_insurance_with_us:** Mean ~4 years. Indicates moderate loyalty. Longer duration means more loyalty

**regular_checkup_lasy_year:** Mean ~1.9 checkups. Reflects moderate proactive care; higher values may indicate stronger health engagement.

**visited_doctor_last_1_year:** Avg ~3.1 visits. Signals potential existing health concerns or better monitoring.

**Cholesterol_level:** Most frequent range: 150–175 mg/dL. Mid-range.

**daily_avg_steps:** Mean ~5,216. Moderately active.

**age:** Mean ~45 years.

**Gender:** Male majority.

**avg_glucose_level:** Mean ~168. High average—potential risk indicator for diabetes, affects premium calculation.

**bmi:** Mean ~31.4.

**smoking_status:** Dominated by “never smoked”.

**Location:** Could reflect regional cost variation.

**weight:** Mean ~84 kg.

**covered_by_any_other_company:** Mostly “No”. 

**Alcohol:** “Rare” is dominant. Low-risk group; could adjust pricing minimally based on category.

**exercise:** “Moderate” dominates. Positive lifestyle indicator—linked to reduced claims risk.

**weight_change_in_last_one_year:** Avg ~2.9 kg. Could indicate lifestyle change, recovery, or instability.

**fat_percentage:** Mean ~28.8%. Higher fat% aligns with elevated BMI—signals cardio-metabolic risk.

**insurance_cost:** Mean ~₹27,147.

## Missing value Imputation

**BMI**

In [ ]:
data['bmi'] = data.groupby(['Gender', 'exercise','age'])['bmi'].transform(lambda x: x.fillna(x.mean()))

**Year_last_admitted**

The idea behind Year Last Admitted is to define what counts as a recent hosipital admission.

Engineering this feature to **Recent Admission** would Flag admissions within certain Range like 2 or 3 years as clinically relavant.

But, that again depends on assumption that missing values in this feature as never admitted which inturn makes this feature very synthetic. 

**Dropping this feature makes more sense.**

In [ ]:
data.drop(["Year_last_admitted"], axis=1, inplace=True)

In [ ]:
data.isnull().sum()

## Checking for Outliers

In [ ]:
import math

numeric_columns = data.select_dtypes(include=np.number).columns.tolist()
num_plots = len(numeric_columns)
cols = 4
rows = math.ceil(num_plots / cols)

plt.figure(figsize=(5 * cols, 4 * rows))

for i, variable in enumerate(numeric_columns):
    plt.subplot(rows, cols, i + 1)
    plt.boxplot(data[variable], whis=1.5)
    plt.title(variable)
    plt.xticks([])  # Optional: remove x-axis ticks to declutter

plt.tight_layout(pad=3.0)
plt.show()


## Exploratory Data Analysis

#### The below functions need to be defined to carry out the Exploratory Data Analysis.

In [ ]:
# function to plot a boxplot and a histogram along the same scale.


def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    """
    Boxplot and histogram combined

    data: dataframe
    feature: dataframe column
    figsize: size of figure (default (12,7))
    kde: whether to the show density curve (default False)
    bins: number of bins for histogram (default None)
    """
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2,  # Number of rows of the subplot grid= 2
        sharex=True,  # x-axis will be shared among all subplots
        gridspec_kw={"height_ratios": (0.25, 0.75)},
        figsize=figsize,
    )  # creating the 2 subplots
    sns.boxplot(
        data=data, x=feature, ax=ax_box2, showmeans=True, color="violet"
    )  # boxplot will be created and a triangle will indicate the mean value of the column
    sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins, palette="winter"
    ) if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2
    )  # For histogram
    ax_hist2.axvline(
        data[feature].mean(), color="green", linestyle="--"
    )  # Add mean to the histogram
    ax_hist2.axvline(
        data[feature].median(), color="black", linestyle="-"
    )  # Add median to the histogram

In [ ]:
# function to create labeled barplots


def labeled_barplot(data, feature, perc=False, n=None):
    """
    Barplot with percentage at the top

    data: dataframe
    feature: dataframe column
    perc: whether to display percentages instead of count (default is False)
    n: displays the top n category levels (default is None, i.e., display all levels)
    """

    total = len(data[feature])  # length of the column
    count = data[feature].nunique()
    if n is None:
        plt.figure(figsize=(count + 1, 5))
    else:
        plt.figure(figsize=(n + 1, 5))

    plt.xticks(rotation=90, fontsize=15)
    ax = sns.countplot(
        data=data,
        x=feature,
        hue=feature,
        palette="Paired",
        legend=False,
        order=data[feature].value_counts().index[:n].sort_values(),
    )

    for p in ax.patches:
        if perc == True:
            label = "{:.1f}%".format(
                100 * p.get_height() / total
            )  # percentage of each class of the category
        else:
            label = p.get_height()  # count of each level of the category

        x = p.get_x() + p.get_width() / 2  # width of the plot
        y = p.get_height()  # height of the plot

        ax.annotate(
            label,
            (x, y),
            ha="center",
            va="center",
            size=12,
            xytext=(0, 5),
            textcoords="offset points",
        )  # annotate the percentage

    plt.show()  # show the plot

In [ ]:
# function to plot stacked bar chart

def stacked_barplot(data, predictor, target):
    """
    Print the category counts and plot a stacked bar chart

    data: dataframe
    predictor: independent variable
    target: target variable
    """
    count = data[predictor].nunique()
    sorter = data[target].value_counts().index[-1]
    tab1 = pd.crosstab(data[predictor], data[target], margins=True).sort_values(
        by=sorter, ascending=False
    )
    print(tab1)
    print("-" * 120)
    tab = pd.crosstab(data[predictor], data[target], normalize="index").sort_values(
        by=sorter, ascending=False
    )
    tab.plot(kind="bar", stacked=True, figsize=(count + 1, 5))
    plt.legend(
        loc="lower left", frameon=False,
    )
    plt.legend(loc="upper left", bbox_to_anchor=(1, 1))
    plt.show()

In [ ]:
### Function to plot distributions

def distribution_plot_wrt_target(data, predictor, target):

    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    target_uniq = data[target].unique()

    axs[0, 0].set_title("Distribution of target for target=" + str(target_uniq[0]))
    sns.histplot(
        data=data[data[target] == target_uniq[0]],
        x=predictor,
        kde=True,
        ax=axs[0, 0],
        color="teal",
    )

    axs[0, 1].set_title("Distribution of target for target=" + str(target_uniq[1]))
    sns.histplot(
        data=data[data[target] == target_uniq[1]],
        x=predictor,
        kde=True,
        ax=axs[0, 1],
        color="orange",
    )

    axs[1, 0].set_title("Boxplot w.r.t target")
    sns.boxplot(data=data, x=target, y=predictor, ax=axs[1, 0], palette="gist_rainbow")

    axs[1, 1].set_title("Boxplot (without outliers) w.r.t target")
    sns.boxplot(
        data=data,
        x=target,
        y=predictor,
        ax=axs[1, 1],
        showfliers=False,
        palette="gist_rainbow",
    )

    plt.tight_layout()
    plt.show()

### Univariate analysis

`years_of_insurance_with_us`

In [ ]:
labeled_barplot(data, "years_of_insurance_with_us")

Observations:
* Peak at 3 Years: Highest concentration is at 3 years (2,990 customers)

`regular_checkup_lasy_year`

In [ ]:
labeled_barplot(data, "regular_checkup_lasy_year")

Observations:
* Most customers don't engage in regular checkups.

`adventure_sports`

In [ ]:
labeled_barplot(data, "adventure_sports")

Observations:
* Most customers don't engage in adventures sports.

`Occupation`

In [ ]:
labeled_barplot(data, "Occupation")

In [ ]:
data.groupby('Occupation')['insurance_cost'].mean().sort_values()

Observations:

* Salaried group:  Lowest cost. Suggests this group may lead more stable.
* Student group: Mid-range. Despite younger age and fewer health issues.
* Business group: Highest cost. May reflect stress, sedentary hours, or inconsistent healthcare habits.

`visited_doctor_last_1_year`

In [ ]:
labeled_barplot(data, "visited_doctor_last_1_year")

`cholesterol_level`

In [ ]:
labeled_barplot(data, "cholesterol_level")

Observations:
* Majority of individuals fall within 125–175 range — relatively safe zone. Encouraging from a population health perspective.

`daily_avg_steps`

In [ ]:
histogram_boxplot(data, "daily_avg_steps", kde=True)

Observations:
* Most customers log between 4,000–6,000 steps daily, creating a dense peak around that zone.
* There's a long tail extending beyond 8,000 steps, but with much lower frequency—very few ultra-active users.

`age`

In [ ]:
histogram_boxplot(data, "age", kde=True)

Observations:
* Median Age: Hovering around 45 years.
* Mostly uniform across 20–70

`heart_decs_history`

In [ ]:
labeled_barplot(data, "heart_decs_history")

Observations:
* Majority of customers have no prior heart conditions

`other_major_decs_history`

In [ ]:
labeled_barplot(data, "other_major_decs_history")

Observations:
* Majority have no record og other major diseases

`Gender`

In [ ]:
labeled_barplot(data, "Gender")

Observations:
* Male group are Dominant in this dataset

`avg_glucose_level`

In [ ]:
histogram_boxplot(data, "avg_glucose_level", kde=True)

Observations:
* The distribution leans right — indicating a concentration of users with above-normal glucose levels.

`bmi`

In [ ]:
histogram_boxplot(data, "bmi", kde=True)

Observations:
* Peak Density Zone: Most individuals fall between 28–33 BMI
* Skewness: Right-skewed, with a long tail stretching beyond 40

`smoking_status`

In [ ]:
labeled_barplot(data, "smoking_status")

`Location`

In [ ]:
labeled_barplot(data, "Location")

`weight`

In [ ]:
histogram_boxplot(data, "weight", kde=True)

Observations:
* Central Peak: Highest density lies around 70–75.
* Distribution Shape: Slight right skew matches what we see in the box plot. Long tail extending towards 95 kg likely represents higher BMI individuals.

`covered_by_any_other_company`

In [ ]:
labeled_barplot(data, "covered_by_any_other_company")

Observations:
* Majority have no alternate coverage.

`Alcohol`

In [ ]:
labeled_barplot(data, "Alcohol")

Observations:
* Majority group—occasional drinkers.
* Non-drinkers—this group could include health-conscious individuals.
* Smaller but crucial group—daily consumption higher risk

`exercise`

In [ ]:
labeled_barplot(data, "exercise")

`weight_change_in_last_one_year`

Observations
* Moderate - Majority group — these individuals likely benefit from cardiovascular protection and may show reduced risk profiles.
* No - higher risk for obesity, chronic illness, and elevated claims.
* Extreme - Enthusiasts — smaller segment engaging in intense activity.

`fat_percentage`

In [ ]:
histogram_boxplot(data, "fat_percentage", kde=True)

Observations:
* Peak Zone: The mode falls around 35% fat, forming the densest cluster.
* Distribution Shape: Right-skewed with a slow taper toward 45%+.

`insurance_cost`

In [ ]:
histogram_boxplot(data, "insurance_cost", kde=True)

Observations:
* Peak Zone: Highest density of insurance cost lies between 20,000 and 30,000
* Distribution Shape: Right-skewed — long tail stretches toward ₹70,000, indicating a small but impactful set of high-risk insurances

### Feature Engineering

In [ ]:
# cholesterol_level
cholesterol_level = { "125 to 150": 0,"150 to 175": 1,"175 to 200": 2,"200 to 225": 3,"225 to 250": 4 }

data["cholesterol_level"] = data["cholesterol_level"].map(cholesterol_level)

#smoking_status
smoking_status = { "never smoked": 0,"formerly smoked": 1,"smokes": 2,"Unknown": 3 } 

data["smoking_status"] = data["smoking_status"].map(smoking_status)

#Alcohol
Alcohol_status = { "No": 0,"Rare": 1,"Daily": 2 } 
data["Alcohol"] = data["Alcohol"].map(Alcohol_status)

# exercise
exercise_status = { "Moderate": 0,"Extreme": 1,"No": 2 } 

data["exercise"] = data["exercise"].map(exercise_status)

### Bivariate analysis

**Correlation Check**

In [ ]:
cols_list = data.select_dtypes(include=np.number).columns.tolist()

plt.figure(figsize=(12, 7))
sns.heatmap(
    data[cols_list].corr(), annot=True, vmin=-1, vmax=1, fmt=".2f", cmap="Spectral"
)
plt.show()

* **Fat Percentage** shows the strongest positive correlation with insurance cost. Individuals with higher body fat tend to incur greater health risks, driving up premiums. It’s a direct indicator of metabolic and cardiovascular strain.

* **Average Glucose Level** also has a substantial positive correlation. Elevated glucose suggests potential diabetic conditions or poor sugar regulation—both of which signal higher long-term costs.

* **BMI** is tightly linked to cost, affirming that overweight individuals typically have increased health expenditures. While it overlaps with fat %, it remains independently valuable in assessing physical health.

* **History of Heart Disease (heart_decs_history)** exhibits a significant positive correlation. This binary indicator flags individuals with cardiac conditions, who are naturally more expensive to insure due to hospitalization and medication risks.

* **Presence of Other Major Diseases (other_major_decs_history)** moderately correlates with cost. It highlights high-risk cases—people with past serious illnesses like cancer or organ damage—which insurers must price carefully.

* **Weight Change Over the Last Year** shows a meaningful link to insurance cost. Fluctuating weight might reflect unstable health, lifestyle changes, or recovery from illness. Such volatility often translates into increased claim likelihood.

* **Age** has a mild correlation with insurance cost. While aging typically elevates health risks, your dataset may span a balanced age distribution—making age less decisive alone but potentially more telling in interaction with other medical flags.

* **Cholesterol Level** is surprisingly less correlated. While medically relevant, its cost impact may be dampened by confounding factors like medication or health-conscious behaviors in higher-risk groups.

* **Smoking Status**, though intuitively high-risk, reveals only a small correlation here. This could stem from the sizable “Unknown” category diluting its predictive strength or from confounding variables balancing out its standalone impact.

* **Alcohol** Consumption Frequency has a modest correlation. Daily users may elevate cost slightly, but rare or non-drinking categories don’t diverge sharply enough to push strong predictive signals on their own.

* **Exercise** Behavior shows a slight negative correlation, hinting that regular activity may help reduce risk—but it lacks strength as an isolated feature. Its full impact likely emerges when paired with BMI, fat %, or glucose.

* **Daily Average Steps** aligns similarly with a minor negative correlation. Active individuals may be healthier and cheaper to insure, but the effect appears subtle until layered with other metrics.

* **Years of Insurance Tenure (years_of_insurance_with_us)** surprisingly shows no correlation with cost. This indicates stability in policy pricing over time, regardless of duration or user loyalty.

* **Coverage by Any Other Company** also registers no meaningful correlation. Having external insurance doesn’t seem to affect the cost assessed in your dataset—possibly due to diverse motivations behind dual coverage.

** mean insurance_cost vs regular_checkup_lasy_year  **

In [ ]:
mean_regular_checkup_lasy_year= data.groupby(['regular_checkup_lasy_year'])[['insurance_cost']].mean()
mean_regular_checkup_lasy_year

In [ ]:
sns.catplot(data=mean_regular_checkup_lasy_year,x="regular_checkup_lasy_year",y="insurance_cost",hue="regular_checkup_lasy_year",kind='bar')
plt.xticks(rotation=90)

* Individuals with 0 Checkups Have the Lowest Mean Cost This cohort likely includes healthier, younger, or risk-averse individuals who don’t visit doctors unless absolutely necessary. Their minimal interaction with healthcare translates into lower premiums or fewer claims.

* 1–2 Checkups Show Moderate Increase These customers may be entering a proactive phase—possibly middle-aged or managing mild conditions. Their cost starts climbing, but not drastically. They represent a transitional risk tier.

* 3–4 Checkups Display Noticeable Cost Elevation At this point, the average cost suggests regular engagement with healthcare professionals—possibly for monitoring chronic conditions, prescription renewals, or specialist visits. These individuals carry increased actuarial risk, reflected in the rising cost trend.

* 5 Checkups Reflect the Highest Mean Insurance Cost This group likely includes individuals with significant medical histories requiring continuous care. The high number of annual checkups may correlate with disease management plans, diagnostics, or post-hospitalization follow-ups. Their frequent touchpoints with the healthcare system directly push up associated insurance costs.

**mean insurance_cost vs adventure_sports**

In [ ]:
mean_adventure_sports= data.groupby(['adventure_sports'])[['insurance_cost']].mean()
mean_adventure_sports

In [ ]:
sns.catplot(data=mean_adventure_sports,x="adventure_sports",y="insurance_cost",hue="adventure_sports",kind='bar')
plt.xticks(rotation=90)

* Non-participants (value = 0) These individuals average around ₹27,000 in insurance cost. Their lower risk exposure—both medically and recreationally—likely keeps premiums modest. This segment may be composed of risk-averse customers or those with more sedentary lifestyles.

* Participants (value = 1) Their mean cost jumps to roughly ₹32,000, signaling that engaging in high-risk recreational activities such as adventure sports drives premiums upward. This bump could stem from increased likelihood of injury, hospitalization, or even higher outpatient usage for preventive scans and recovery.

**mean insurance_cost vs Occupation**

In [ ]:
mean_Occupation= data.groupby(['Occupation'])[['insurance_cost']].mean()
mean_Occupation

In [ ]:
sns.catplot(data=mean_Occupation,x="Occupation",y="insurance_cost",hue="Occupation",kind='bar')
plt.xticks(rotation=90)

* Business Professionals (~₹27,270) This group leads in insurance cost. Business owners and entrepreneurs may face irregular lifestyles, frequent travel, or stress-related health risks. Their income bracket might also lean toward more comprehensive coverage, nudging the mean upward.

* Students (~₹27,145) Surprisingly close to the business segment, student costs may reflect academic stress, sedentary habits, or irregular health behavior. Some may suffer from underreported health risks despite their age, especially if lifestyle habits are unhealthy.

* Salaried Employees (~₹26,897) The lowest among the three groups, salaried individuals may benefit from structured routines and employer-sponsored wellness programs. Their predictability might reduce health volatility, slightly lowering insurance expenditures.

**mean insurance_cost vs cholesterol_level**

In [ ]:
mean_cholesterol_level= data.groupby(['cholesterol_level'])[['insurance_cost']].mean()
mean_cholesterol_level

In [ ]:
sns.catplot(data=mean_cholesterol_level,x="cholesterol_level",y="insurance_cost",hue="cholesterol_level",kind='bar')
plt.xticks(rotation=90)

* Cholesterol Level 0 (125–150 range) Mean cost is around ₹27,163. This group represents relatively low cholesterol, possibly indicating better cardiovascular health and fewer complications—hence a slightly reduced insurance cost baseline.

* Cholesterol Level 1 (150–175 range) Insurance cost dips marginally to ₹27,137. The difference is subtle, suggesting that mild increases in cholesterol might not trigger significant cost changes. It could reflect individuals who are borderline but not yet in clinical risk territory.

* Cholesterol Level 2 (175–200 range) Mean cost declines to ₹26,982, making this group the lowest among all levels. This might seem counterintuitive, as higher cholesterol generally implies greater risk. It could hint at better lifestyle compensation (e.g., younger age, active behavior) within this band, or sampling dynamics affecting cost outcomes.

* Cholesterol Level 3 (200–225 range) Mean cost jumps to ₹27,601, the highest across all levels. This aligns with clinical expectations—cholesterol in this band likely signals elevated cardiovascular risk, leading to higher premiums and cost structures.

**insurance_cost vs daily_avg_steps**

In [ ]:
data['daily_avg_steps'] = pd.cut(data['daily_avg_steps'], bins=[0, 4000, 8000, 12000], labels=['Low', 'Moderate', 'High'])

In [ ]:
# exercise
activity_band= { "Low": 0,"Moderate": 1,"High": 2 } 

data["daily_avg_steps"] = data["daily_avg_steps"].map(activity_band)

In [ ]:
mean_daily_avg_steps= data.groupby(['daily_avg_steps'])[['insurance_cost']].mean()
mean_daily_avg_steps

In [ ]:
sns.catplot(data=mean_daily_avg_steps,x="daily_avg_steps",y="insurance_cost",hue="daily_avg_steps",kind='bar')
plt.xticks(rotation=90)

* Low Activity (0–4000 steps/day → Band 0) With a mean insurance cost of ₹27,091, this group reflects a more sedentary lifestyle. Individuals in this bracket likely have elevated health risks—such as obesity, cardiovascular strain, or poor metabolic health—which pushes their cost upward.

* Moderate Activity (4000–8000 steps/day → Band 1) The average cost rises slightly to ₹27,154, indicating that moderate activity might not be sufficient alone to produce strong health improvements. These individuals could be transitioning from sedentary behavior or managing early-stage conditions.

* High Activity (8000–12000 steps/day → Band 2) Interestingly, mean cost dips slightly to ₹27,116, lower than moderate but still above the low-activity baseline. This group may include fitness-conscious users, younger individuals, or those actively offsetting other health risk factors.

**insurance_cost vs heart_decs_history**

In [ ]:
mean_heart_decs_history= data.groupby(['heart_decs_history'])[['insurance_cost']].mean()
mean_heart_decs_history

In [ ]:
sns.catplot(data=mean_heart_decs_history,x="heart_decs_history",y="insurance_cost",hue="heart_decs_history",kind='bar')
plt.xticks(rotation=90)

* The above graph tells Heart Disease History customers experince higher insurance cost

**insurance_cost vs other_major_decs_history**

In [ ]:
mean_other_major_decs_history= data.groupby(['other_major_decs_history'])[['insurance_cost']].mean()
mean_other_major_decs_history

In [ ]:
sns.catplot(data=mean_other_major_decs_history,x="other_major_decs_history",y="insurance_cost",hue="other_major_decs_history",kind='bar')
plt.xticks(rotation=90)

* The above graph tells tOther Major Disease History customers experince higher insurance cost

**insurance_cost vs Gender**

In [ ]:
mean_Gender= data.groupby(['Gender'])[['insurance_cost']].mean()
mean_Gender

In [ ]:
sns.catplot(data=mean_Gender,x="Gender",y="insurance_cost",hue="Gender",kind='bar')
plt.xticks(rotation=90)

* Even though Data is baised towards Male Gender then Mean insurance costs are almost equal this tells female customers in this data are having critical health conditions

**insurance_cost vs avg_glucose_level**

In [ ]:
def glucose_risk(val):
    if val <= 110:
        return 0  # Safe
    elif val <= 160:
        return 1  # Borderline
    else:
        return 2  # High risk

data['avg_glucose_level'] = data['avg_glucose_level'].apply(glucose_risk)


In [ ]:
data['avg_glucose_level'].value_counts()

In [ ]:
mean_avg_glucose_level= data.groupby(['avg_glucose_level'])[['insurance_cost']].mean()
mean_avg_glucose_level

In [ ]:
sns.catplot(data=mean_avg_glucose_level,x="avg_glucose_level",y="insurance_cost",hue="avg_glucose_level",kind='bar')
plt.xticks(rotation=90)

* Glucose Level 0 (Low Range) With a mean insurance cost of ₹27,186, this group has the highest average cost. That might seem counterintuitive, but this band could include individuals with hypoglycemia or underlying health monitoring behaviors. Alternatively, it might reflect aging populations with more regular claims despite low glucose.

* Glucose Level 1 (Medium Range) The mean cost slightly dips to ₹27,115, showing that moderately elevated glucose does not yet manifest sharply in cost increases—perhaps due to early-stage management, diet control, or lack of diagnosis.

* Glucose Level 2 (High Range) Mean insurance cost rises again to ₹27,144. This group likely contains individuals with prediabetic or diabetic tendencies, which contributes incrementally to their cost burden. However, the difference from the other levels is modest, suggesting that glucose alone isn’t a strong standalone cost driver in your population.

**insurance_cost vs bmi**

In [ ]:
def bmi_band(bmi):
    if bmi < 25:
        return 0
    elif bmi < 35:
        return 1
    else:
        return 2
data['bmi'] = data['bmi'].apply(bmi_band)


In [ ]:
data['bmi'].value_counts()

In [ ]:
mean_bmi= data.groupby(['bmi'])[['insurance_cost']].mean()
mean_bmi

In [ ]:
sns.catplot(data=mean_bmi,x="bmi",y="insurance_cost",hue="bmi",kind='bar')
plt.xticks(rotation=90)

* BMI Band 0 (<25) Mean insurance cost is ₹27,445 — the highest across all bands. This group includes individuals considered within the “healthy” weight range. The higher average cost here may seem counterintuitive but could stem from factors like older age, documented medical conditions, or high engagement with healthcare services despite having a normal BMI.

* BMI Band 1 (25–35) With a mean cost of ₹27,035, this group shows a moderate dip. These individuals may be classified as overweight but not obese. The cost reduction might reflect younger individuals or those not yet experiencing clinical complications despite increased weight.

* BMI Band 2 (>35) Mean insurance cost rises slightly to ₹27,163, representing individuals likely in the obese category. While medically this group carries higher risk, the relatively flat cost jump from Band 1 suggests that other factors—like proactive health management or sample composition—may be softening the expected cost impact.

**insurance_cost vs smoking_status**

In [ ]:
data['smoking_status'].value_counts()

In [ ]:
mean_smoking_status= data.groupby(['smoking_status'])[['insurance_cost']].mean()
mean_smoking_status

In [ ]:
sns.catplot(data=mean_smoking_status,x="smoking_status",y="insurance_cost",hue="smoking_status",kind='bar')
plt.xticks(rotation=90)

* Never Smoked (status = 0) Individuals in this group show a mean insurance cost of ₹27,080 — slightly higher than some other categories. This might baised data of non smokers in the data.

* Formerly Smoked (status = 1) With a mean cost of ₹27,052, this group may represent individuals recovering from prior smoking behavior. They might still be experiencing residual health effects (e.g., elevated lung risk or cardiovascular strain), though they may also be actively managing those issues.

* Currently Smokes (status = 2) The average cost here drops a bit to ₹27,031. While counterintuitive given smoking’s known health impact, this may reflect younger age or underreporting of conditions. Alternatively, it could be a signal that some high-risk individuals haven't yet manifested major chronic claims.

* Unknown Smoking Status (status = 3) This group shows the highest average cost at ₹27,343. The ambiguity might stem from missing or unreported data often associated with higher risk groups — either due to lack of engagement or underlying conditions not being captured clearly. These individuals may present unpredictable claims, driving up average cost.

**insurance_cost vs Alcohol**

In [ ]:
data['Alcohol'].value_counts()

In [ ]:
mean_Alcohol= data.groupby(['Alcohol'])[['insurance_cost']].mean()
mean_Alcohol

In [ ]:
sns.catplot(data=mean_Alcohol,x="Alcohol",y="insurance_cost",hue="Alcohol",kind='bar')
plt.xticks(rotation=90)

* Customers who comsume Alcohol Daily has the average cost of insurance equal to higher counts of customers in other 2 categories 

**insurance_cost vs exercise**

In [ ]:
data['exercise'].value_counts()

In [ ]:
mean_exercise= data.groupby(['exercise'])[['insurance_cost']].mean()
mean_exercise

In [ ]:
sns.catplot(data=mean_exercise,x="exercise",y="insurance_cost",hue="exercise",kind='bar')
plt.xticks(rotation=90)

* Exercise Level 0 (Moderate)

Mean Cost: ₹27,273 — the highest among all exercise tiers.

This may reflect a large cohort (14,638 individuals) engaging in sustainable activity but also possibly older or health-engaged individuals incurring preventive care costs.

Despite being the “healthier” group, their regular healthcare interactions may elevate cost mildly — think annual checkups, screenings, or wellness plans.

* Exercise Level 1 (Extreme)

Mean Cost: ₹27,052

These individuals likely represent fitness enthusiasts. The cost dip suggests they’re healthier overall, though occasional injuries or higher engagement may still register in claims.

Sample size is moderate (5,248), enough to stabilize cost interpretation.

* Exercise Level 2 (No Exercise)

Mean Cost: ₹26,884 — surprisingly the lowest mean insurance cost.

However, this group includes only 5,114 individuals. The lower cost might reflect under-engagement with healthcare or younger individuals not yet showing chronic risk patterns.

Could also signal hidden risk — sedentary behavior that hasn’t yet led to claim-worthy illness.

**insurance_cost vs fat_percentage**

In [ ]:
def fat_band(val):
    if val < 25:
        return 0
    elif val < 35:
        return 1
    else:
        return 2
data['fat_percentage'] = data['fat_percentage'].apply(fat_band)


In [ ]:
data['fat_percentage'].value_counts()

In [ ]:
mean_fat_percentage= data.groupby(['fat_percentage'])[['insurance_cost']].mean()
mean_fat_percentage

In [ ]:
sns.catplot(data=mean_fat_percentage,x="fat_percentage",y="insurance_cost",hue="fat_percentage",kind='bar')
plt.xticks(rotation=90)

* Band 0 (<25% fat)

Mean Cost: ₹27,290

Count: 9,129 individuals

This “Lean” group surprisingly carries the highest average insurance cost. Possible reasons? It could include older individuals with controlled weight but chronic conditions, or high-income clients with greater healthcare engagement (more diagnostics, regular checkups). Their large count further amplifies total insurance expenditure.

* Band 1 (25–35% fat)

Mean Cost: ₹27,263

Count: 6,739 individuals

These are “Moderate Risk” clients — the lowest count but still showing a high mean cost. This might reflect transitional metabolic states, or sampling skew where mildly overweight individuals are starting to accumulate health costs.

* Band 2 (>35% fat)

Mean Cost: ₹26,919

Count: 9,141 individuals

The “High Risk” band has the largest population and lowest mean insurance cost. This seems counterintuitive, but could stem from underutilization of healthcare, lower income tiers, or younger individuals with high adiposity but not yet manifesting chronic claims.

## Data Preprocessing

### Data Preparation One Hot Encoding

In [ ]:
# creating dummy variables
data = pd.get_dummies(
    data,
    columns=data.select_dtypes(include=["object", "category"]).columns.tolist(),
    drop_first=True
)
data.head()

In [ ]:
# converting the input attributes into float type for modeling
data = data.astype(float)
data.head()

## Clustering

## Using RFE Model to pick top features in the data

## Removing Target variable insurance_cost from clustering

In [ ]:
# Exclude insurance_cost from inputs
X = data.drop(columns=['insurance_cost'])

In [ ]:
y = data['insurance_cost']

In [ ]:
# # RFE setup
# model = RandomForestRegressor(random_state=42)
# rfe = RFE(estimator=model, n_features_to_select=15)
# rfe.fit(X, y)

# # Extract top 10 features
# top_features = X.columns[rfe.support_]
# print("Top 15 features:", top_features.tolist())

## Segmenting into 2 types of groups

In [ ]:
numerical_cols = [
    'years_of_insurance_with_us', 'cholesterol_level', 'age',
    'avg_glucose_level', 'bmi', 'weight', 'weight_change_in_last_one_year',
    'fat_percentage'
]

categorical_cols = [
    'regular_checkup_lasy_year', 'visited_doctor_last_1_year',
    'smoking_status', 'Alcohol', 'exercise',
    'Gender_Male', 'covered_by_any_other_company_Y'
]


In [ ]:
sample = data[numerical_cols + categorical_cols].copy()
sample.to_clipboard()

In [ ]:
subset_df = data[numerical_cols + categorical_cols].dropna().copy()
subset_df[categorical_cols] = subset_df[categorical_cols].astype(str)


In [ ]:
# Assigning Subset_df dataframe to another one
X = data[numerical_cols + categorical_cols]

In [ ]:
# from kmodes.kprototypes import KPrototypes

# # Convert to numpy matrix
# X_matrix = subset_df.to_numpy()

# # Fit K-Prototypes (start with 4 clusters — can adjust using elbow or silhouette)
# kproto = KPrototypes(n_clusters=4, random_state=42)
# clusters = kproto.fit_predict(X_matrix, categorical=[subset_df.columns.get_loc(col) for col in categorical_cols])

# # Assign cluster labels
# subset_df['cluster'] = clusters


In [ ]:
# def plot_clustered_bars(df, cluster_col, categorical_feature):
#     plt.figure(figsize=(8, 5))
#     sns.countplot(data=df, x=categorical_feature, hue=cluster_col, palette='Set2')
#     plt.title(f'Distribution of {categorical_feature} across Clusters')
#     plt.xlabel(categorical_feature)
#     plt.ylabel('Count')
#     plt.legend(title='Cluster')
#     plt.tight_layout()
#     plt.show()

# # Apply to all categorical features
# for col in categorical_cols:
#     plot_clustered_bars(subset_df, 'cluster', col)


## Recomedations and Conclusions

## Cluster 0 — Health-Conscious but High-Cost
* Traits: Moderate exercise, frequent doctor visits, fewer smokers/alcohol users, mostly male, limited external coverage

* Interpretation: Proactive with healthcare — routine checkups and visits may drive higher utilization

* Recommendation: Consider wellness rewards or preventive engagement discounts. Encourage digital health tracking to manage claims more efficiently.

## Cluster 1 — Low Engagement, Silent Risk
* Traits: Minimal checkups, low doctor visits, sedentary lifestyle with higher alcohol scores, low external coverage

* Interpretation: May avoid the healthcare system until late-stage events — potential for delayed chronic emergence

* Recommendation: Deploy awareness and outreach campaigns, possibly nudging via mobile or employer plans. Explore incentive-based health activation to shift behavior early.

## Cluster 2 — Lean, Fitness-Forward Males
* Traits: Extreme or no exercise, mostly male, lower engagement, split alcohol habits, more externally insured

* Interpretation: Younger and leaner group, possibly underutilizing your plans due to partial coverage

* Recommendation: Offer customized fitness-linked plans or optional bundling features to bring them deeper into engagement. Watch for claim spikes via metabolic flags or orthopedic risks.

## Cluster 3 — Data-Ambiguous but Potentially High-Risk

* Traits: Many smoking status unknowns, high alcohol prevalence, neutral exercise, gender-balanced, external coverage present

* Interpretation: Missing or ambiguous traits suggest hidden risks — this group may be cost-volatile and harder to profile

* Recommendation: Invest in better data capture, targeted surveys, and possibly flag for underwriting review. Consider semi-automated monitoring or claim pattern alerts.

# **Model Building**

# **Parametric models**

## 1. OLS Model

In [ ]:
# For randomized data splitting
from sklearn.model_selection import train_test_split

# To build linear regression_model
import statsmodels.api as sm

# To check model performance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score 

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
# let's add the intercept to data
X = sm.add_constant(X)

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
# splitting the data in 70:30 ratio for train to test data

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

In [ ]:
print("Number of rows in train data =", x_train.shape[0])
print("Number of rows in test data =", x_test.shape[0])

In [ ]:
nan_rows = X[X.isnull().any(axis=1)]
nan_rows

### Model Building - Linear Regression

In [ ]:
olsmodel = sm.OLS(y_train, x_train).fit()
print(olsmodel.summary())

### Model Performance Check

In [ ]:
# function to compute adjusted R-squared
def adj_r2_score(predictors, targets, predictions):
    r2 = r2_score(targets, predictions)
    n = predictors.shape[0]
    k = predictors.shape[1]
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))


# function to compute MAPE
def mape_score(targets, predictions):
    return np.mean(np.abs(targets - predictions) / targets) * 100


# function to compute different metrics to check performance of a regression model
def model_performance_regression(model, predictors, target):
    """
    Function to compute different metrics to check regression model performance

    model: regressor
    predictors: independent variables
    target: dependent variable
    """

    # predicting using the independent variables
    pred = model.predict(predictors)

    r2 = r2_score(target, pred)  # to compute R-squared
    adjr2 = adj_r2_score(predictors, target, pred)  # to compute adjusted R-squared
    rmse = np.sqrt(mean_squared_error(target, pred))  # to compute RMSE
    mae = mean_absolute_error(target, pred)  # to compute MAE
    mape = mape_score(target, pred)  # to compute MAPE

    # creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {
            "RMSE": rmse,
            "MAE": mae,
            "R-squared": r2,
            "Adj. R-squared": adjr2,
            "MAPE": mape,
        },
        index=[0],
    )

    return df_perf

In [ ]:
# checking model performance on train set (seen 70% data)
print("Training Performance\n")
olsmodel_train_perf = model_performance_regression(olsmodel, x_train, y_train)
olsmodel_train_perf['Model']= 'OLS Model'
olsmodel_train_perf

### Checking Linear Regression Assumptions

We will be checking the following Linear Regression assumptions:

1. **No Multicollinearity**

2. **Linearity of variables**

3. **Independence of error terms**

4. **Normality of error terms**

5. **No Heteroscedasticity**

### TEST FOR MULTICOLLINEARITY

* **General Rule of thumb**:
    - If VIF is between 1 and 5, then there is low multicollinearity.
    - If VIF is between 5 and 10, we say there is moderate multicollinearity.
    - If VIF is exceeding 10, it shows signs of high multicollinearity.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor


def checking_vif(predictors):
    vif = pd.DataFrame()
    vif["feature"] = predictors.columns

    # calculating VIF for each feature
    vif["VIF"] = [
        variance_inflation_factor(predictors.values, i)
        for i in range(len(predictors.columns))
    ]
    return vif

In [ ]:
checking_vif(x_train)

* There are no multiple columns with very high VIF values, indicating no presence of multicollinearity
* We will systematically drop numerical columns with VIF > 5
* We will ignore the VIF values for dummy variables and the constant (intercept)

**Now we'll check the rest of the assumptions on *olsmodel*.**

2. **Linearity of variables**

3. **Independence of error terms**

4. **Normality of error terms**

5. **No Heteroscedasticity**

In [ ]:
# let us create a dataframe with actual, fitted and residual values
df_pred = pd.DataFrame()

df_pred["Actual Values"] = y_train  # actual values
df_pred["Fitted Values"] = olsmodel.fittedvalues  # predicted values
df_pred["Residuals"] = olsmodel.resid  # residuals

df_pred.head()

In [ ]:
# let's plot the fitted values vs residuals

sns.residplot(
    data=df_pred, x="Fitted Values", y="Residuals", color="purple", lowess=True
)
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Fitted vs Residual plot")
plt.show()

* The scatter plot shows the distribution of residuals (errors) vs fitted values (predicted values).

* If there exist any pattern in this plot, we consider it as signs of non-linearity in the data and a pattern means that the model doesn't capture non-linear effects.

* **We see no pattern in the plot above. Hence, the assumptions of linearity and independence are satisfied.**

### TEST FOR NORMALITY

In [ ]:
sns.histplot(data=df_pred, x="Residuals", kde=True)
plt.title("Normality of residuals")
plt.show()

- The histogram of residuals does have a bell shape.
- Let's check the Q-Q plot.

In [ ]:
import pylab
import scipy.stats as stats

stats.probplot(df_pred["Residuals"], dist="norm", plot=pylab)
plt.show()

In [ ]:
stats.shapiro(df_pred["Residuals"])

- Since p-value < 0.05, the residuals are not normal as per the Shapiro-Wilk test.
- Strictly speaking, the residuals are not normal.
- However, as an approximation, we can accept this distribution as close to being normal.
- **So, the assumption is satisfied.**

In [ ]:
import statsmodels.stats.api as sms
from statsmodels.compat import lzip

name = ["F statistic", "p-value"]
test = sms.het_goldfeldquandt(df_pred["Residuals"], x_train)
lzip(name, test)

**Since p-value > 0.05, we can say that the residuals are homoscedastic. So, this assumption is satisfied.**

## Predictions on test data

In [ ]:
# predictions on the test set
pred = olsmodel.predict(x_test)

df_pred_test = pd.DataFrame({"Actual": y_test, "Predicted": pred})
df_pred_test.sample(10, random_state=1)

In [ ]:
# checking model performance on test set (seen 30% data)
print("Test Performance\n")
olsmodel_final_test_perf = model_performance_regression(
    olsmodel, x_test, y_test
)
olsmodel_final_test_perf['Model']= 'OLS Model'
olsmodel_final_test_perf

* The model is able to explain ~94% of the variation in the data

## 2. LASSO Model

In [ ]:
from sklearn.linear_model import Lasso

In [ ]:
lasso_model = Lasso(alpha=2.0)  
lasso_model.fit(x_train, y_train)

In [ ]:
print("Lasso Coefficients:", lasso_model.coef_)
print("Intercept:", lasso_model.intercept_)

In [ ]:
# checking model performance on train set (seen 70% data)
print("Training Performance\n")
lasso_model_train_perf = model_performance_regression(lasso_model, x_train, y_train)
lasso_model_train_perf['Model']= 'LASSO Regression Model'
lasso_model_train_perf

## Predictions on test data

In [ ]:
# predictions on the test set
pred = lasso_model.predict(x_test)

df_pred_test = pd.DataFrame({"Actual": y_test, "Predicted": pred})
df_pred_test.sample(10, random_state=1)

In [ ]:
# checking model performance on test set (seen 30% data)
print("Test Performance\n")
lasso_model_final_test_perf = model_performance_regression(
    lasso_model, x_test, y_test
)
lasso_model_final_test_perf['Model']= 'LASSO Regression Model'
lasso_model_final_test_perf

## 3. Ridge Regression Model

In [ ]:
from sklearn.linear_model import Ridge

ridge_model = Ridge(alpha=1.0)
ridge_model.fit(x_train, y_train)

print("Ridge Coefficients:", ridge_model.coef_)
print("Intercept:", ridge_model.intercept_)


In [ ]:
# checking model performance on train set (seen 70% data)
print("Training Performance\n")
ridge_model_train_perf = model_performance_regression(ridge_model, x_train, y_train)
ridge_model_train_perf['Model']= 'Ridge Regression Model'
ridge_model_train_perf

## Predictions on test data

In [ ]:
# predictions on the test set
pred = ridge_model.predict(x_test)

df_pred_test = pd.DataFrame({"Actual": y_test, "Predicted": pred})
df_pred_test.sample(10, random_state=1)

In [ ]:
# checking model performance on test set (seen 30% data)
print("Test Performance\n")
ridge_model_final_test_perf = model_performance_regression(
    ridge_model, x_test, y_test
)
ridge_model_final_test_perf['Model']= 'Ridge Regression Model'
ridge_model_final_test_perf

## 4. Polynomial Regression Model (Using Pipeline)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline


In [ ]:
# Degree 2 for quadratic; tune as needed
poly_model = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
poly_model.fit(x_train, y_train)

print("Polynomial Coefficients:", poly_model.named_steps['linearregression'].coef_)
print("Intercept:", poly_model.named_steps['linearregression'].intercept_)


In [ ]:
# checking model performance on train set (seen 70% data)
print("Training Performance\n")
poly_model_train_perf = model_performance_regression(poly_model, x_train, y_train)
poly_model_train_perf['Model'] = 'Polynomial Regression Model'
poly_model_train_perf

## Predictions on test data

In [ ]:
# predictions on the test set
pred = poly_model.predict(x_test)

df_pred_test = pd.DataFrame({"Actual": y_test, "Predicted": pred})
df_pred_test.sample(10, random_state=1)

In [ ]:
# checking model performance on test set (seen 30% data)
print("Test Performance\n")
poly_model_final_test_perf = model_performance_regression(
    poly_model, x_test, y_test
)
poly_model_final_test_perf['Model'] = 'Polynomial Regression Model'
poly_model_final_test_perf

# **Model Performance Comparison on Training data between Parametric Models**

In [ ]:
models_train_comp_df = pd.concat(
    [
        olsmodel_train_perf,
        lasso_model_train_perf,
        ridge_model_final_test_perf,
        poly_model_train_perf
    ],
    axis=0,
    ignore_index=True
)
models_train_comp_df.set_index('Model')
print("Training performance comparison:")
models_train_comp_df

# **Model Performance Comparison on Testing data between Parametric Models**

In [ ]:
models_test_comp_df = pd.concat(
    [
        olsmodel_final_test_perf,
        lasso_model_final_test_perf,
        ridge_model_train_perf,
        poly_model_final_test_perf
    ],
    axis=0,
    ignore_index=True
)
models_test_comp_df.set_index('Model')
print("Testing performance comparison:")
models_test_comp_df

# Non-Parametric Models

In [ ]:
# Splitting data into training, validation and test set:
# first we split data into 2 parts, say temporary and test

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

# then we split the temporary set into train and validation

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=1, stratify=y_temp
)
print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
print("Number of rows in train data =", X_train.shape[0])
print("Number of rows in validation data =", X_val.shape[0])
print("Number of rows in test data =", X_test.shape[0])

## 1. Random Forest Model

In [ ]:
# Random Forest for Regression
from sklearn.ensemble import RandomForestRegressor

# XGBoost for Regression
from xgboost import XGBRegressor

# AdaBoost for Regression
from sklearn.ensemble import AdaBoostRegressor

# Support Vector Machine for Regression
from sklearn.svm import SVR


In [ ]:
models = []  # Empty list to store all the models

models.append(("Random forest", RandomForestRegressor(random_state=1)))
models.append(("XGB", XGBRegressor(random_state=1)))
models.append(("Adaboost", AdaBoostRegressor(random_state=1)))
models.append(('SVR', SVR()))

In [ ]:
print("\nTraining Performance:\n")
for name, model in models:
    model.fit(X_train, y_train)
    scores = model_performance_regression(model, X_train, y_train)
    print(f"{name}:\n{scores}\n")

print("\nValidation Performance:\n")
for name, model in models:
    model.fit(X_train, y_train)
    scores = model_performance_regression(model, X_val, y_val)
    print(f"{name}:\n{scores}\n")

In [ ]:
print("\nTraining and Validation Performance Difference:\n")

performance_list = []

for name, model in models:
    model.fit(X_train, y_train)
    train_perf = model_performance_regression(model, X_train, y_train)
    val_perf = model_performance_regression(model, X_val, y_val)

    # Add model name info
    train_perf["Model"] = name

    performance_list.extend([train_perf])

    # Metric differences
    diff_rmse  = train_perf["RMSE"].values[0] - val_perf["RMSE"].values[0]
    diff_mae   = train_perf["MAE"].values[0] - val_perf["MAE"].values[0]
    diff_r2    = train_perf["R-squared"].values[0] - val_perf["R-squared"].values[0]
    diff_adjr2 = train_perf["Adj. R-squared"].values[0] - val_perf["Adj. R-squared"].values[0]
    diff_mape  = train_perf["MAPE"].values[0] - val_perf["MAPE"].values[0]

    print(f"{name}:")
    print(f"  RMSE  - Train: {train_perf['RMSE'].values[0]:.2f}, Validation: {val_perf['RMSE'].values[0]:.2f}, Diff: {diff_rmse:.2f}")
    print(f"  MAE   - Train: {train_perf['MAE'].values[0]:.2f}, Validation: {val_perf['MAE'].values[0]:.2f}, Diff: {diff_mae:.2f}")
    print(f"  R²    - Train: {train_perf['R-squared'].values[0]:.2f}, Validation: {val_perf['R-squared'].values[0]:.2f}, Diff: {diff_r2:.2f}")
    print(f"  Adj R²- Train: {train_perf['Adj. R-squared'].values[0]:.2f}, Validation: {val_perf['Adj. R-squared'].values[0]:.2f}, Diff: {diff_adjr2:.2f}")
    print(f"  MAPE  - Train: {train_perf['MAPE'].values[0]:.2f}%, Validation: {val_perf['MAPE'].values[0]:.2f}%, Diff: {diff_mape:.2f}%\n")


In [ ]:
# Concatenate all into one DataFrame
models_train_perf_df = pd.concat(performance_list, axis=0).reset_index(drop=True)

models_train_perf_df.set_index('Model')

## Models like Random Forest, XGB, Adaboost are good with MAPE but the models might be overfitted. This raises the need for Hyperparameter Tuning all the models.

## Hyperparameter Tuning

### Tuning Random Forest Regresion model with Original data

In [ ]:
%%time

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Base model
Model = RandomForestRegressor(random_state=1)

# Parameter grid for Random Forest
param_grid = {
    "n_estimators": np.arange(50, 200, 25),
    "max_depth": [None, 3, 5, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["auto", "sqrt", "log2"]
}

# Using R² for optimization but store all scores after tuning
def custom_scorer(estimator, X, y):
    scores = model_performance_regression(estimator, X, y)
    return scores["R-squared"].values[0]  

# Tuning
randomized_cv = RandomizedSearchCV(
    estimator=Model,
    param_distributions=param_grid,
    n_iter=50,
    scoring=custom_scorer,
    cv=5,
    n_jobs=-1,
    random_state=1
)

randomized_cv.fit(X_train, y_train)

# Best model
rf_best_model = randomized_cv.best_estimator_

# Evaluate all metrics using your custom function
Tuned_RandomForest_train_scores = model_performance_regression(rf_best_model, X_train, y_train)
Tuned_RandomForest_train_scores['Model'] = 'Tuned RandomForest'
val_scores   = model_performance_regression(rf_best_model, X_val, y_val)

# Display full results
print("\n Final Scores for Best Estimator:")
print("Training Set:")
print(Tuned_RandomForest_train_scores)

print("\nValidation Set:")
print(val_scores)

### Tuning xgboost Regressor model with Original data

In [ ]:

# Base model
Model = XGBRegressor(random_state=1)

# Parameter grid for XGBoost
param_grid = {
    "n_estimators": np.arange(50, 200, 25),
    "max_depth": [3, 5, 7, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.6, 0.8, 1.0]
}
# Custom scorer wrapping your full performance function
def custom_scorer(estimator, X, y):
    scores = model_performance_regression(estimator, X, y)
    return scores["R-squared"].values[0]  

# Randomized search setup
randomized_cv = RandomizedSearchCV(
    estimator=Model,
    param_distributions=param_grid,
    n_iter=50,
    scoring=custom_scorer,
    cv=5,
    n_jobs=-1,
    random_state=1
)

# Fit search
randomized_cv.fit(X_train, y_train)

# Retrieve and evaluate best model
xgboost_best_model = randomized_cv.best_estimator_
Tuned_xgboost_train_scores = model_performance_regression(xgboost_best_model, X_train, y_train)
Tuned_xgboost_train_scores['Model'] = 'Tuned XGBoost'
val_scores = model_performance_regression(xgboost_best_model, X_val, y_val)

# Display full score comparison
print("Best Parameters Found:")
print(randomized_cv.best_params_)
print("\n Performance Metrics — Best XGBoost Estimator")
print("Training:")
print(Tuned_xgboost_train_scores)
print("\nValidation:")
print(val_scores)


### Tuning Adaboost Regressor model with Original data

In [ ]:
# Base AdaBoost model
Model = AdaBoostRegressor(random_state=1)

# Parameter grid
param_grid = {
    "n_estimators": np.arange(50, 200, 25),
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.5, 1.0],
    "loss": ["linear", "square", "exponential"]
}

# Custom scoring wrapper
def custom_scorer(estimator, X, y):
    scores = model_performance_regression(estimator, X, y)
    return scores["R-squared"].values[0]  

# Hyperparameter tuning via RandomizedSearchCV
randomized_cv = RandomizedSearchCV(
    estimator=Model,
    param_distributions=param_grid,
    n_iter=50,
    scoring=custom_scorer,
    cv=5,
    n_jobs=-1,
    random_state=1
)

# Fit
randomized_cv.fit(X_train, y_train)

# Best model and full evaluation
Adaboost_best_model = randomized_cv.best_estimator_
Tuned_Adaboost_train_scores = model_performance_regression(Adaboost_best_model, X_train, y_train)
Tuned_Adaboost_train_scores['Model'] = 'Tuned Adaboost'
val_scores   = model_performance_regression(Adaboost_best_model, X_val, y_val)

# Display results
print("Best Parameters:")
print(randomized_cv.best_params_)
print("\n Performance Metrics — Best AdaBoost Model")
print("Training:")
print(Tuned_Adaboost_train_scores)
print("\nValidation:")
print(val_scores)

### Tuning SVR Regressor model with Original data

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

svr_pipeline = make_pipeline(StandardScaler(), SVR())

# Parameter grid for SVR
param_grid = {
    "svr__kernel": ["rbf", "linear"],  
    "svr__C": [1, 10],
    "svr__gamma": ["scale", 0.1],
    "svr__epsilon": [0.1, 0.5]
}


def custom_scorer(estimator, X, y):
    scores = model_performance_regression(estimator, X, y)
    return scores["R-squared"].values[0] 

# Randomized search
randomized_cv = RandomizedSearchCV(
    estimator=svr_pipeline,
    param_distributions=param_grid,
    n_iter=50,
    scoring=custom_scorer,
    cv=5,
    n_jobs=-1,
    random_state=1
)

# Fitting the tuner
randomized_cv.fit(X_train, y_train)

# Best model and full evaluation
svr_best_model = randomized_cv.best_estimator_
Tuned_svr_train_scores = model_performance_regression(svr_best_model, X_train, y_train)
Tuned_svr_train_scores['Model'] = 'Tuned SVR'
val_scores = model_performance_regression(svr_best_model, X_val, y_val)

# Display results
print("Best Parameters:")
print(randomized_cv.best_params_)
print("\n Performance Metrics — Best SVR Model")
print("Training:")
print(Tuned_svr_train_scores)
print("\nValidation:")
print(val_scores)


# **Model Performance Comparison on Training data between Non Parametric Models**

In [ ]:
tuned_models_train_perf_df = pd.concat(
    [
        Tuned_RandomForest_train_scores,
        Tuned_xgboost_train_scores,
        Tuned_Adaboost_train_scores,
        Tuned_svr_train_scores,
    ],
    axis=0,
    ignore_index=True
)
tuned_models_train_perf_df.set_index('Model')

In [ ]:
models_train_perf_df = pd.concat([models_train_perf_df,tuned_models_train_perf_df],
                                 axis=0,
                                 )

models_train_perf_df.set_index('Model')

In [ ]:
# Let's check the performance on Random Forest test set
rf_test = model_performance_regression(rf_best_model, X_test, y_test)
rf_test['Model'] = 'Tuned Random Forest'

In [ ]:
# Let's check the performance on XGBoost Model test set
xgboost_test = model_performance_regression(xgboost_best_model, X_test, y_test)
xgboost_test['Model'] = 'Tuned XGBoost'

In [ ]:
# Let's check the performance on Adaboost Model test set
Adaboost_test = model_performance_regression(Adaboost_best_model, X_test, y_test)
Adaboost_test['Model'] = 'Tuned Adaboost'

In [ ]:
# Let's check the performance on SVR Model test set
svr_test = model_performance_regression(svr_best_model, X_test, y_test)
svr_test['Model'] = 'Tuned SVR'

# **Model Performance Comparison on Testing data between Non Parametric Models**

In [ ]:
tuned_models_test_perf_df = pd.concat(
    [
        rf_test,
        xgboost_test,
        Adaboost_test,
        svr_test,
    ],
    axis=0,
    ignore_index=True
)
tuned_models_test_perf_df.set_index('Model')

# **Model Performance Comparison on Training data between All Models**

In [ ]:
All_models_train_comp_df = pd.concat([models_train_comp_df,tuned_models_train_perf_df],axis=0)
All_models_train_comp_df.set_index('Model')

In [ ]:
# Sorting dataframe with lowest MAPE at top
sorted_df_train_comp_df = All_models_train_comp_df.sort_values(by="MAPE", ascending=True)


sorted_df_train_comp_df = sorted_df_train_comp_df.set_index('Model')

In [ ]:
sorted_df_train_comp_df

# **Model Performance Comparison on Testing data between All Models**

In [ ]:
All_models_test_comp_df = pd.concat([models_test_comp_df,tuned_models_test_perf_df],axis=0)

In [ ]:
All_models_test_comp_df.set_index('Model')

In [ ]:
# Sorting dataframe with lowest MAPE at top
sorted_df_test_comp_df = All_models_test_comp_df.sort_values(by="MAPE", ascending=True)

sorted_df_test_comp_df = sorted_df_test_comp_df.set_index('Model')

In [ ]:
sorted_df_test_comp_df

- ## **All models has given MAPE less than 15% on test data compared with Train data**
- This performance is in line with what we achieved with this model on the train and validation sets
- ## **So, picking top 2 models to check feature importances**

## Feature Importance of Tuned Random Forest Model

In [ ]:
feature_names = X_train.columns
importances = rf_best_model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(12, 12))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

## Feature Importance of Tuned XG Boost Model

In [ ]:
feature_names = X_train.columns
importances = xgboost_best_model.feature_importances_
indices = np.argsort(importances)

plt.figure(figsize=(12, 12))
plt.title("Feature Importances")
plt.barh(range(len(indices)), importances[indices], color="violet", align="center")
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel("Relative Importance")
plt.show()

# **Interpretation of the Most Optimum model**

### **Tuned XGBoost:**
* The Tuned XGBoost model stands out as the most accurate and business-aligned choice in lineup. With a MAPE of just 12.26%, it offers exceptionally tight forecasting performance.
* Its R-squared value of 0.9527 confirms that it captures over 95% of the variance in the target variable, signifying strong feature.
* The model also posted the lowest RMSE and MAE, smoother performance across diverse customer profiles.
* In terms of feature behavior, XGBoost places overwhelming emphasis on “weight”. Other features like coverage by another insurance company and regular health checkups register minor but intuitive signals, highlighting behavioral and contextual flags.

### **Tuned Random Forest:**
* The Tuned XGBoost model stands out as the slightly lesser accurate than Tuned XG Boost model. With a MAPE of just 13.76%, it offers exceptionally tight forecasting performance.
* Its R-squared value of 0.9457 confirms that it captures over 94% of the variance in the target variable, this model still delivers highly dependable performance — just a notch below XGBoost.
* The slightly higher RMSE and MAE suggest broader prediction intervals but remain well within acceptable thresholds.
* In terms of feature behavior, Random Forest truly shines is in its clarity of feature importances. Not only does it reaffirm **weight** as the primary driver ~80% importance, but it also surfaces **weight change in the last year** as a meaningful secondary signal ~10%. Other features like **checkups**, **age**, **tenure**, and **lifestyle indicators** contribute minimally